# Food Detection with YOLO (UEC FOOD 100 subset)

主食・主菜・副菜・汁物の12品目を対象に、YOLOで複数品目の物体検出を行います。
分類演習(`food_CNN_classification.ipynb`)が「1品目だけ切り出した画像」を扱ったのに対し、
こちらは給食トレイのような**1枚の写真に複数品目が写っている**、より実際の場面に近い画像を扱います。

事前準備:
- UEC FOOD 100 データセット一式(1/～100/フォルダ、multiple_food.txt など)をGoogleドライブにアップロード
- `convert_uecfood100_to_yolo.py` と `category_ja_utf8.txt` も同じドライブ上に配置

## 1. Googleドライブのマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 環境に合わせて書き換えてください
base_dir = '/content/drive/MyDrive/food_yolo/'
dataset_root = base_dir + 'UECFOOD100/'
output_dir = base_dir + 'yolo_data/'
category_names_path = base_dir + 'category_ja_utf8.txt'
convert_script_path = base_dir + 'convert_uecfood100_to_yolo.py'

## 2. ライブラリのインストール

In [ ]:
!pip install ultralytics pillow -q

## 3. データ変換(bb_info.txt → YOLO形式)

以前用意した `convert_uecfood100_to_yolo.py` を、選んだ12品目を指定して実行します。
同じ写真が複数カテゴリに重複している場合(例: ごはん+味噌汁)も、自動的に1枚の画像・
1つのラベルファイルにまとめられます。

In [ ]:
CATEGORIES = "1,36,46,55,56,60,63,67,69,70,87,90"

!python "{convert_script_path}" \
    --dataset-root "{dataset_root}" \
    --output-dir "{output_dir}" \
    --categories "{CATEGORIES}" \
    --category-names "{category_names_path}" \
    --val-fraction 0.2 \
    --seed 42

In [ ]:
# 出力内容の確認
import os

for split in ("train", "val"):
    n_images = len(os.listdir(output_dir + f"images/{split}"))
    n_labels = len(os.listdir(output_dir + f"labels/{split}"))
    print(f"{split}: images={n_images}, labels={n_labels}")

print()
with open(output_dir + "data.yaml", encoding="utf-8") as f:
    print(f.read())

## 4. 学習(Fine-tuning)

In [ ]:
from ultralytics import YOLO

# nano(最小・最速)モデルをベースに、今回の12クラスでfine-tuning
model = YOLO("yolo11n.pt")

results = model.train(
    data=output_dir + "data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="food_yolo",
)

## 5. 評価(mAP・Precision・Recallなど)

Roboflow側の結果(mAP@50, Precision, Recall, F1)と同じ指標で比較できるよう、検証データで評価します。

In [ ]:
metrics = model.val()

print(f"mAP@50:    {metrics.box.map50:.3f}")
print(f"mAP@50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

## 6. 推論と可視化

学習したモデルで、実際の写真から複数品目を検出してみます。
Roboflow側で作った `detect_and_show()` と同じ使い勝手になるよう、
同名の関数として揃えています(こちらは `detect_and_show_yolo` という名前です)。

In [ ]:
import matplotlib.pyplot as plt

def detect_and_show_yolo(image_path, confidence=0.4):
    result = model.predict(image_path, conf=confidence, verbose=False)[0]

    annotated = result.plot()  # ultralyticsが描画済みの画像(BGR)を返す
    annotated_rgb = annotated[..., ::-1]  # BGR -> RGB

    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(annotated_rgb)
    ax.axis('off')
    plt.show()
    return result

# 使い方
result = detect_and_show_yolo("test_photo.jpg")

## 7. 2つのモデルを見比べる

同じ写真に対して、Roboflow側のモデル(`detect_and_show`)と、
今回のYOLOモデル(`detect_and_show_yolo`)を並べて実行すれば、
「少数の自分の写真だけで学習したモデル」と「既存の大きいデータセットで学習したモデル」の
違いを直接見比べられます。Roboflow側のセルをこのノートブックにコピーしてから、
以下のように両方を呼び出してみてください。

```python
print("--- Roboflow (自分の写真のみで学習) ---")
detect_and_show("test_photo.jpg")

print("--- YOLO (既存の大きいデータセットで学習) ---")
detect_and_show_yolo("test_photo.jpg")
```

## 8. 学習済みの重みをドライブに保存

`best.pt` は今のままだとColab自体のディスク(`/content/...`)にあるだけなので、セッションが切れると消えてしまいます。
学生用の演習(`food_YOLO_exercise.ipynb`)で使えるように、ドライブにコピーしておきます。

In [ ]:
!cp /content/runs/detect/food_yolo/weights/best.pt \
    "/content/drive/MyDrive/Colab Notebooks/food_yolo/best.pt"

print("コピー完了。food_YOLO_exercise.ipynb からはこのパスを読み込みます。")